# Notebook 5: Reading & Writing Data (05_File_Operations.ipynb)

Welcome back, friends! In the last notebook, we learned what Pandas is. Today, we are going to learn how to **read data from different file formats** into Pandas and **write data back** into those formats. 

In real life, data is never sitting inside our Python code. It comes from different sources like Excel sheets from the finance team, CSV files from web downloads, JSON from mobile apps, or huge SQL databases. Let us master how to handle them easily!

---

## 1. Real-World Analogy: The Logistics and Filing Office

Think of Pandas as a highly efficient **Office Clerk** sitting at a desk. 
* **Reading Data:** This is like the clerk receiving folders from different couriers (CSV, Excel, JSON) and spreading them flat on the desk as a neatly formatted register book (DataFrame) so it can be easily read.
* **Writing Data:** This is like the clerk finishing the work, putting the papers back into specific types of file boxes (like a fast binary Parquet box or an Excel file), and sending them back to different departments.

---

## 2. File Path Handling (`pathlib`)
Before reading files, we must find where they are located. In the old days, people used simple string names, but that creates errors between Windows and Mac laptops. Now, we use the modern `pathlib` library. It acts like a **smart GPS navigation system** for your computer's folders.

In [1]:
from pathlib import Path
import pandas as pd

# Setting up a clean folder for our data files
data_dir = Path("sample_data")
data_dir.mkdir(exist_ok=True)

print(f"Data directory created successfully at: {data_dir.absolute()}")

Data directory created successfully at: C:\Users\skmah\Downloads\sample_data


---

## 3. CSV Files
**CSV (Comma Separated Values)** is the simplest format. It is like a basic plain-text notepad where each value is separated by a comma.

In [2]:
# Simple Example: Creating sample student data
data = {'Roll_No': [101, 102, 103], 'Name': ['Amit', 'Priya', 'Rahul'], 'Marks': [85, 92, 78]}
df = pd.DataFrame(data)

# Defining file path
csv_path = data_dir / "students.csv"

# 1. Writing to CSV
df.to_csv(csv_path, index=False) # index=False avoids writing row numbers as a separate column
print("Saved CSV file!")

# 2. Reading from CSV
df_csv = pd.read_csv(csv_path)
df_csv.head()

Saved CSV file!


,Roll_No,Name,Marks
0,101,Amit,85
1,102,Priya,92
2,103,Rahul,78


---

## 4. Excel Files
Excel (`.xlsx`) files are widely preferred by business managers because they contain pretty colors, tabs, and multi-sheet layouts.

In [3]:
excel_path = data_dir / "students.xlsx"

# Writing to Excel
df.to_excel(excel_path, sheet_name="Term_1_Marks", index=False)
print("Saved Excel file!")

# Reading from Excel
df_excel = pd.read_excel(excel_path, sheet_name="Term_1_Marks")
df_excel.head()

Saved Excel file!


,Roll_No,Name,Marks
0,101,Amit,85
1,102,Priya,92
2,103,Rahul,78


---

## 5. JSON Files
**JSON (JavaScript Object Notation)** looks exactly like a Python dictionary. It is used extensively when data travels across mobile applications and website APIs.

In [ ]:
json_path = data_dir / "students.json"

# Writing to JSON
df.to_json(json_path, orient="records", indent=4)
print("Saved JSON file!")

# Reading from JSON
df_json = pd.read_json(json_path)
df_json.head()

---

## 6. Parquet & Pickle Files (Advanced Binary Storage)
* **Parquet:** A columnar storage format that compresses massive files beautifully. It is heavily utilized in Big Data frameworks.
* **Pickle:** A native Python method to freeze and save any object (like dataframes or ML models) directly to disk.

In [ ]:
parquet_path = data_dir / "students.parquet"
pickle_path = data_dir / "students.pkl"

# Writing Parquet and Pickle
df.to_parquet(parquet_path)
df.to_pickle(pickle_path)
print("Saved Parquet and Pickle files!")

# Reading back
df_parquet = pd.read_parquet(parquet_path)
df_pickle = pd.read_pickle(pickle_path)
print("Successfully read binary formats!")

---

## 7. SQL Tables
In professional production setups, data stays inside relational databases like SQLite, MySQL, or PostgreSQL. We can fetch data directly using standard SQL queries.

In [ ]:
import sqlite3

# Creating an in-memory temporary database connection
conn = sqlite3.connect(":memory:")

# Writing a DataFrame straight into a SQL table
df.to_sql("student_table", conn, index=False, if_exists="replace")
print("Saved data directly inside SQL table!")

# Reading from SQL table using a query query
df_sql = pd.read_sql_query("SELECT * FROM student_table WHERE Marks > 80", conn)
conn.close()
df_sql.head()

---

## 8. Reading Multiple Files Automatically
Imagine a retailer sending you separate monthly transaction CSV logs. You do not want to write `read_csv` manually 12 times. We use Python's `glob` tracking or `pathlib` pattern matching to join them collectively.

In [ ]:
# Let us create 3 distinct regional sales CSV files as a mock example
pd.DataFrame({'Store': ['Delhi'], 'Sales': [50000]}).to_csv(data_dir / "sales_delhi.csv", index=False)
pd.DataFrame({'Store': ['Mumbai'], 'Sales': [65000]}).to_csv(data_dir / "sales_mumbai.csv", index=False)
pd.DataFrame({'Store': ['Bangalore'], 'Sales': [72000]}).to_csv(data_dir / "sales_bangalore.csv", index=False)

# Dynamically search and bind all sales files together
all_sales_files = list(data_dir.glob("sales_*.csv"))
print(f"Discovered target files: {all_sales_files}")

# List comprehension to read all CSVs and concatenate them together
combined_df = pd.concat([pd.read_csv(file) for file in all_sales_files], ignore_index=True)
combined_df

---

## 9. Business Example
**Scenario:** You run an e-commerce startup in India. The Operations Team provides daily delivery tracking logs in a standard CSV file, while the Finance Team updates accounting transaction fees inside a JSON master file. You need to combine both formats to find real profit margins and export the final summary as an executive Excel file.

In [ ]:
# 1. Create mock files for operations and finance
ops_df = pd.DataFrame({'OrderID': ['TXN1', 'TXN2'], 'Status': ['Delivered', 'In-Transit']})
fin_df = pd.DataFrame({'OrderID': ['TXN1', 'TXN2'], 'Amount_INR': [1200, 4500]})

ops_df.to_csv(data_dir / "daily_ops.csv", index=False)
fin_df.to_json(data_dir / "daily_fin.json", orient="records")

# 2. Read them in
loaded_ops = pd.read_csv(data_dir / "daily_ops.csv")
loaded_fin = pd.read_json(data_dir / "daily_fin.json")

# 3. Merge them and export for business managers
business_report = pd.merge(loaded_ops, loaded_fin, on="OrderID")
business_report.to_excel(data_dir / "Executive_Business_Report.xlsx", index=False)
print("Business Report generated completely!")
business_report

### How this represents the business concept:
In day-to-day corporate environments, information exists in siloed formats. This example mirrors a real corporate lifecycle: ingestion of low-level logs from multiple files (`read_csv`, `read_json`), processing values inside a centralized hub, and serving it inside a familiar corporate interface (`to_excel`).

---

## 10. AI/ML Use Case
**Scenario:** You are training a machine learning model to predict house loan defaults using a dataset of 5 million historical customers. Reading 5 million lines from a raw CSV file takes too long and slows down your training pipelines. To optimize this, you must read the data once, clean it, save it into highly optimized **Parquet** format, and see how fast it loads.

In [ ]:
import time
import numpy as np

# Create a large sample dataset representing 100,000 customers
large_data = {
    'CustomerID': np.arange(100000, 200000),
    'Credit_Score': np.random.randint(300, 850, size=100000),
    'Income_INR': np.random.randint(20000, 200000, size=100000),
    'Default_Risk': np.random.choice([0, 1], size=100000)
}
large_df = pd.DataFrame(large_data)

# Save in both CSV and Parquet
large_df.to_csv(data_dir / "ml_large_dataset.csv", index=False)
large_df.to_parquet(data_dir / "ml_large_dataset.parquet")

# Measure read time for CSV
start_time = time.time()
pd.read_csv(data_dir / "ml_large_dataset.csv")
csv_duration = time.time() - start_time

# Measure read time for Parquet
start_time = time.time()
pd.read_parquet(data_dir / "ml_large_dataset.parquet")
parquet_duration = time.time() - start_time

print(f"Time taken to read CSV dataset: {csv_duration:.4f} seconds")
print(f"Time taken to read compressed Parquet dataset: {parquet_duration:.4f} seconds")
print(f"Parquet is roughly {csv_duration / parquet_duration:.1f}x faster!")

### How this represents the AI/ML concept:
When building machine learning workflows, model data loading loops occur thousands of times during hyperparameter tuning experiments. Using native text-based formats like CSV slows down feature engineering. By migrating file operations to columnar binary formats like **Parquet**, data scientists reduce disk footprint and achieve rapid sub-second initialization speeds.